<a href="https://colab.research.google.com/github/esla-boom/devf/blob/main/Copia_de_AngelV_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00


In [ ]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print('Cliente groq inicializado correctamente.')

Cliente groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasificar el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llego tarde, pero el producto es excelente.' respuesta breve y corta"
response_grande = client.chat.completions.create(
    model="allam-2-7b",
    messages=[
        {
            "role": "user",
            "content": prompt_zero_shot
        }
    ]
)
print("\nRespuesta zero shot:\n", response_grande.choices[0].message.content)



Respuesta zero shot:
 Mixto 


In [ ]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasificar el sentimiento de esta reseña en Positivo, Negativo o Mixto.
Reseña: "Me encantó, llego rápido y en perfecto estado."
Sentimiento: Positivo
Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo
Reseña: "El envío llego tarde, pero el producto es excelente."
Sentimiento:"""

response_grande = client.chat.completions.create(
    model="allam-2-7b",
    messages=[
        {
            "role": "user",
            "content": prompt_few_shot
        }
    ]
)
print("\nRespuesta few shot:\n", response_grande.choices[0].message.content)



Respuesta few shot:
 Mixto 


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)
problem = ("Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
"ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tarda el segundo tren en "
"alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final.")

response = client.chat.completions.create(
    model="allam-2-7b",
    messages=[
        {
            "role": "user",
            "content": problem
        }
    ]
)
print("\nRespuesta chain-of-thought:\n", response.choices[0].message.content)


Respuesta chain-of-thought:
 Para calcular cuánto tarda el segundo tren en alcanzar al primero, debemos considerar las distancias que cada tren recorre a medida que avanzan juntos.

1. El primer tren sale a 80 km/h y tarda dos horas en llegar, pero ya contamos estas dos horas en el tiempo que tiene que pasar el segundo tren para alcanzarlo. Por lo tanto, no contamos estos datos al calcular el tiempo que lleva el segundo tren para unirse al primero.

2. Desde que ambos trenes comenzaren juntos, están recorriendo juntos una distancia total de 80 km/h (primera velocidad) más 120 km/h (segunda velocidad), las dos horas que tardó el primero en llegar.

3. Digamos que después de la primera hora, los dos trenes ya han recorrido juntos 80 + 120 = 200 km, porque cada uno está moviéndose igual velocidad que el otro desde la tardanza del inicio.

4. Después de dos horas (contando la primera travesía), juntos han recorrido 80 + 2 * 120 = 320 km, porque el primer tren vuelve a embarcar y a juntars

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = ("¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 30 de agosto del"
"2026? Respuesta muy breve y corta")

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-safeguard-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_desconocido
        }
    ]
)
print("\nRespuesta:\n", response_alucinacion.choices[0].message.content)


Respuesta:
 Lo siento, no tengo esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
"Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
"empaque original.",
"Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
"envío de regreso.",
"Los productos en oferta o liquidación no son elegibles para devolución, solo para "
"cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante


In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG


modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [

    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "

    "empaque original.",

    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "

    "envío de regreso.",

    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "

    "cambio de talla."

]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [ ]:
# Leer API key, instalar e importar librerías


In [ ]:
# Definir la lista documentos y generar sus embeddings


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [ ]:
# Definir la función buscar_fragmento


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [ ]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [ ]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [ ]:
# Mostrar ambas respuestas para comparar
